In [1]:
# %%
import warnings
warnings.filterwarnings('ignore')

# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
from pathlib import Path
import os
import sys
from collections import defaultdict

# Statistical libraries
from scipy.stats import spearmanr, pearsonr
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy import stats
import scipy.sparse as sp
from statsmodels.stats.multitest import multipletests

# Machine learning
from sklearn.linear_model import LinearRegression, LogisticRegression, RidgeCV
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import r2_score, accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Specialized libraries
import scanpy as sc
from adjustText import adjust_text
from tqdm import tqdm

# Read in the anndata object
import anndata as ad
from pathlib import Path
import numpy as np
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
from scipy.stats import spearmanr, pearsonr
import scipy.sparse as sp
from scipy.cluster.hierarchy import linkage, dendrogram
from matplotlib.ticker import ScalarFormatter
import scanpy as sc
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import cross_val_score, KFold, StratifiedKFold
from sklearn.metrics import r2_score, accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
from collections import defaultdict
import warnings
from adjustText import adjust_text
import torch 
warnings.filterwarnings('ignore')

from scipy import stats
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

# Import LeafletFA differential splicing code
# Define module paths
src_path = "/gpfs/commons/home/kisaev/LeafletFA/src/"

# Add to sys.path if not already present
if src_path not in sys.path:
    sys.path.append(src_path)

# Import custom modules
import BetaDirichletFactor.differential_splicing as ds

# Import utility functions - simple direct import
sys.path.append('/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Multi_Species_Splicing_Foundation/shared_utils/')
from utils import *
from figure_plotting import *

# Import all functions from /gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/shared_utils/atse_viz.py
from atse_viz import *
import gffutils

db_file_mouse = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/ATSE_mapper/genomes/lr_GRCm38.db"
db_mouse = gffutils.FeatureDB(db_file_mouse, keep_order=True)

Added /gpfs/commons/home/kisaev/LeafletFA-utils to sys.path
Visualization imports successful!


In [2]:
# Load the aging gene datasets
aging_genes_mouse = "/gpfs/commons/groups/knowles_lab/Karin/HAGR/GenAge/model/genage_models.csv"
aging = pd.read_csv(aging_genes_mouse)
aging = aging[aging["organism"] == "Mus musculus"]

In [3]:
cell_type_mapping_file="/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Mouse_Splicing_Foundation/Figures/LeafletFA_Supplemental_Table_1.xlsx" 
# read the excel file 
cell_type_mapping = pd.read_excel(cell_type_mapping_file)
cell_type_mapping.head()

,cell_id,cell_name,cell_ontology_class,broad_cell_type,tissue,tissue_celltype,dataset
0,A10_B000120,A10_B000120,basal epithelial cell of tracheobronchial tree,Epithelial,Trachea,Trachea_Epithelial,TMS
1,A10_B000126,A10_B000126,bulge keratinocyte,Epithelial,Skin,Skin_Keratinocyte,TMS
2,A10_B000127,A10_B000127,myeloid cell,Immune,SCAT,SCAT_Myeloid,TMS
3,A10_B000166,A10_B000166,basal cell,Epithelial,Mammary_Gland,Mammary_Epithelial,TMS
4,A10_B000169,A10_B000169,endothelial cell of coronary artery,Endothelial,Heart,Heart_VascEndo,TMS


In [31]:
# sample a random row from cell_type_mapping
cell_type_mapping.sample(1)
# =============================================================================

,cell_id,cell_name,cell_ontology_class,broad_cell_type,tissue,tissue_celltype,dataset
20564,D3_B000814,D3_B000814,mesenchymal stem cell of adipose,Stem/Progenitor,GAT,GAT_MSC,TMS


In [24]:
cell_type_mapping[cell_type_mapping["tissue_celltype"] == "Tongue_Keratinocyte"]

,cell_id,cell_name,cell_ontology_class,broad_cell_type,tissue,tissue_celltype,dataset
64,A10_B001218,A10_B001218,keratinocyte,Epithelial,Tongue,Tongue_Keratinocyte,TMS
65,A10_B001220,A10_B001220,basal cell of epidermis,Epithelial,Tongue,Tongue_Keratinocyte,TMS
83,A10_B001392,A10_B001392,basal cell of epidermis,Epithelial,Tongue,Tongue_Keratinocyte,TMS
113,A10_B002441,A10_B002441,basal cell of epidermis,Epithelial,Tongue,Tongue_Keratinocyte,TMS
114,A10_B002443,A10_B002443,basal cell of epidermis,Epithelial,Tongue,Tongue_Keratinocyte,TMS
...,...,...,...,...,...,...,...
100454,P9_B002755,P9_B002755,basal cell of epidermis,Epithelial,Tongue,Tongue_Keratinocyte,TMS
100459,P9_B002777,P9_B002777,basal cell of epidermis,Epithelial,Tongue,Tongue_Keratinocyte,TMS
100460,P9_B002780,P9_B002780,keratinocyte,Epithelial,Tongue,Tongue_Keratinocyte,TMS
100509,P9_D041894,P9_D041894,basal cell of epidermis,Epithelial,Tongue,Tongue_Keratinocyte,TMS


In [32]:
# =============================================================================
# CONFIGURATION - EDIT THESE VALUES ONLY
# =============================================================================

# Base directories
BASE_DIR = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION"
RESULTS_BASE_DIR = "/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Mouse_Splicing_Foundation/model_train/MOUSE_FOUNDATION/results"
GE_ANNDATA_scVI_PATH = f"{BASE_DIR}/scVI/ge_adata_with_scvi_model_latent_20_20000_2025-10-09.h5ad"

# Reference files
AGING_GENES_PATH = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/27857814"
RBP_FILE_PATH = "/gpfs/commons/groups/knowles_lab/Karin/VanNostrand_2020_supptable1_41586_2020_2077_MOESM3_ESM.xlsx"

ATSE_FILE_PATH = (
    f"{BASE_DIR}/ATSE_mapper/ATSE_files/MOUSE_FOUNDATION_ATSE_FILE_unanno_also_2025-10-01_21-36-40.txt.gz"
)

# Load ATSE file
atse_df = pd.read_csv(ATSE_FILE_PATH, sep="\t")
atse_df["junction_annotation"] = "Novel_SS" 
atse_df.loc[atse_df["perfect_match_5_prime"].notna(), "junction_annotation"] = "5_prime_annotated"
atse_df.loc[atse_df["perfect_match_3_prime"].notna(), "junction_annotation"] = "3_prime_annotated"
atse_df.loc[atse_df["perfect_match_5_prime"].notna() & atse_df["perfect_match_3_prime"].notna(), "junction_annotation"] = "Both_SS_annotated"

# Load splicing data
ge_adata = ad.read_h5ad(GE_ANNDATA_scVI_PATH)
print(f"Done reading the ge_adata with scVI...")

# If ge_adata.var["gene_name"] is not in ge_adata.var_names, then add it
if "gene_name" not in ge_adata.var.columns:
    ge_adata.var["gene_name"] = ge_adata.var_names

# If ge_adata.obs doesn't have cell_id make it from cell_id_clean
if "cell_id" not in ge_adata.obs.columns:
    ge_adata.obs["cell_id"] = ge_adata.obs["cell_id_clean"]

Done reading the ge_adata with scVI...


In [33]:
# Load aging gene lists
aging_genes_mouse, aging_genes_human = load_aging_genes(AGING_GENES_PATH)
    
# Load RBP genes
rbps = load_rbp_genes(RBP_FILE_PATH)
rbps["mouse_gene_name"] = rbps["mouse_gene_name"].str.upper()
aging_genes_mouse = [g.upper() for g in aging_genes_mouse]

# if "mouse.id" is in splice_adata.obs rename it to donor_id 
print(f"Renaming mouse.id to donor_id in splice_adata.obs")

# Update gene annotations
ge_adata.var["RBP_gene"] = ge_adata.var["gene_name"].isin(rbps["mouse_gene_name"])
ge_adata.var["Aging_gene"] = ge_adata.var["gene_name"].isin(aging_genes_mouse)
rbps = rbps["mouse_gene_name"]

Loaded 330 aging-related genes (mouse)
Loaded 330 aging-related genes (human)
Loaded 356 RNA binding proteins
Renaming mouse.id to donor_id in splice_adata.obs


In [34]:
# do sanity check using /gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/27857814 genes 
sanity_aging = pd.read_csv("/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/TabulaSenis/27857814", sep="\t")
sanity_aging["global_aging_genes"] = sanity_aging["global_aging_genes"].str.upper()
sanity_aging.head()

,global_aging_genes,prop_of_up_regulated_tissue_cell_types,gag_score_coef
0,0910001L09RIK,1.000000,0.010753
1,2010107E04RIK,0.000000,-0.005263
2,2310044H10RIK,0.956063,0.010753
3,2900097C17RIK,0.185792,-0.005263
4,9530068E07RIK,0.000000,-0.005263


In [35]:
ge_adata.var_names = ge_adata.var["gene_name"]
ge_adata.obsm["X_normalized_scVI_linear"].columns = ge_adata.var["gene_name"]

In [36]:
# 1. Create a DataFrame from the expression matrix
expr_df = pd.DataFrame(
    ge_adata.obsm["X_normalized_scVI_linear"], 
    index=ge_adata.obs_names, 
    columns=ge_adata.var_names
)

# 2. Map Column Headers from Gene ID to Gene Name
# matches IDs (10162) to Names (Snapc1)
id_to_name_map = ge_adata.var['gene_name'].to_dict()
expr_df.columns = expr_df.columns.map(id_to_name_map)

# 1. FIX: Set the gene name column as the index so .loc can find them
# We create a copy to avoid modifying the original variable accidentally
sanity_aging_indexed = sanity_aging.set_index('global_aging_genes')

# 2. Re-calculate the intersection with the new index
common_genes = expr_df.columns.intersection(sanity_aging_indexed.index)
print(f"Number of genes used for scoring: {len(common_genes)}")

# 3. Calculate Score using the indexed dataframe
# Now .loc[common_genes] will work because the index contains the names
aging_scores = expr_df[common_genes].dot(sanity_aging_indexed.loc[common_genes, 'gag_score_coef'])

# 4. Save
ge_adata.obs['GAG_score'] = aging_scores

Number of genes used for scoring: 250


In [37]:
# Main parameters to change
MODEL_TRAIN_DATE = "2025-11-19"
MODEL_ANALYSIS_DATE = "2025-11-22"
PARAM_ID = 11

# =============================================================================
# AUTO-GENERATED PATHS - DON'T EDIT BELOW THIS LINE
# =============================================================================

# Core result directories
PARAM_RESULTS_DIR = f"{RESULTS_BASE_DIR}/{MODEL_TRAIN_DATE}/{MODEL_ANALYSIS_DATE}/param_id_{PARAM_ID}"
DATA_DIR = f"{PARAM_RESULTS_DIR}/data"

# Model outputs
MODEL_OUTPUTS_DIR = f"{BASE_DIR}/Leaflet/leafletFAmodel/{MODEL_TRAIN_DATE}"
MODEL_PATH = f"{MODEL_OUTPUTS_DIR}/run_{PARAM_ID}/leafletfa_model.pkl.gz"

# Main data files from downstream analysis of model 
SPLICE_ADATA_PATH = f"{DATA_DIR}/splice_adata_PHI_psi_var_obs.h5ad"
PI_VALUES_PATH = f"{DATA_DIR}/PI_values.npy"
DIFF_SPL_PATH = f"{DATA_DIR}/differential_splicing_results.csv"

# Output directory for current analysis
OUTPUT_DIR = f"{PARAM_RESULTS_DIR}/analysis_outputs"

In [38]:
# =============================================================================
# LOAD DATA USING CONFIGURED PATHS
# =============================================================================

print(f"Loading data for param_id {PARAM_ID} from {MODEL_TRAIN_DATE}")
print(f"Data directory: {DATA_DIR}")

# Load main datasets
splice_adata = ad.read_h5ad(SPLICE_ADATA_PATH)
splice_adata.var["junction_id_index"] = np.arange(splice_adata.shape[1])
splice_adata.var["num_junctions"].value_counts()

Loading data for param_id 11 from 2025-11-19
Data directory: /gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Mouse_Splicing_Foundation/model_train/MOUSE_FOUNDATION/results/2025-11-19/2025-11-22/param_id_11/data


num_junctions
2    24864
3    24249
4    11844
5     9145
Name: count, dtype: int64

#### Map the updated cell type labels onto splice_adata via cell_ids/cell_names

In [39]:
# remove all the old colummns in splice_adata.obs other than cell_id and cell_name that are in cell_type_mapping
# find columns in cell_type_mapping that are in splice_adata.obs
columns_to_drop = [col for col in cell_type_mapping.columns if col in splice_adata.obs.columns] 
# remove cell_id and cell_name from columns_to_drop
columns_to_drop.remove("cell_id")
columns_to_drop.remove("cell_name")
print(columns_to_drop)

# remove columns from splice_adata.obs
splice_adata.obs.drop(columns=columns_to_drop, inplace=True)
# reset index of splice_adata.obs
splice_adata.obs.reset_index(drop=True, inplace=True)
splice_adata.obs = pd.merge(splice_adata.obs, cell_type_mapping, on=["cell_id", "cell_name"], how="left")

# check that splice_adata.obs.index is exactly the same as splice_adata.obs["cell_id_index"]
assert np.all(splice_adata.obs.index == splice_adata.obs["cell_id_index"]), "Index of splice_adata.obs is not exactly the same as cell_id_index"
assert np.all(ge_adata.obs["cell_id"].values == splice_adata.obs["cell_id"].values), "Cell IDs in ge_adata and splice_adata do not match or are not in the same order."
splice_adata.obs_names = splice_adata.obs["cell_id"]

['cell_ontology_class', 'broad_cell_type', 'tissue', 'dataset']


In [40]:
print(splice_adata.obs["tissue"].value_counts(), "\n", splice_adata.obs["broad_cell_type"].value_counts(), "\n", splice_adata.obs["medium_cell_type"].value_counts(), "\n", splice_adata.obs["tissue_celltype"].value_counts())

tissue
Brain              70840
Marrow             11772
Heart               7856
Large_Intestine     5633
Lung                4542
Skin                4310
Thymus              3525
Limb_Muscle         3418
Spleen              3164
Tongue              3093
GAT                 2942
SCAT                2884
MAT                 2547
Pancreas            2433
Trachea             2302
Liver               2107
BAT                 1862
Mammary_Gland       1853
Kidney              1544
Bladder             1529
Diaphragm           1529
Aorta                630
Name: count, dtype: int64 
 broad_cell_type
Neuron_Excitatory    36239
Immune               22584
Epithelial           20179
Neuron_Inhibitory    17512
Stem/Progenitor      15173
Microglia            12042
Endothelial           7953
Stromal               5859
Muscle                2470
Glia                  1955
Neuron_TMS             349
Name: count, dtype: int64 
 medium_cell_type
Cortical excitatory neuron      30868
Inhibitory neuron  

In [41]:
# if "mouse.id" is in splice_adata.obs rename it to donor_id 
if "mouse.id" in splice_adata.obs.columns:
    print(f"Renaming mouse.id to donor_id in splice_adata.obs")
    splice_adata.obs.rename(columns={"mouse.id": "donor_id"}, inplace=True)

# print how many cells are in splice_adata
print(f"There are {len(splice_adata.obs)} cells in splice_adata")

# print how many cells are in ge_adata
print(f"There are {len(ge_adata.obs)} cells in ge_adata")

There are 142315 cells in splice_adata
There are 142315 cells in ge_adata


In [42]:
pi = np.load(PI_VALUES_PATH)
leaflet_model = load_model(MODEL_PATH)
diff_spl = pd.read_csv(DIFF_SPL_PATH)

print(f"Loaded splice_adata: {splice_adata.shape}")
print(f"Loaded PI values: {pi.shape}")

Loading model to device: cpu
Loaded splice_adata: (142315, 70102)
Loaded PI values: (20,)


In [43]:
# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

Output directory: /gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Mouse_Splicing_Foundation/model_train/MOUSE_FOUNDATION/results/2025-11-19/2025-11-22/param_id_11/analysis_outputs


In [44]:
splice_adata

AnnData object with n_obs × n_vars = 142315 × 70102
    obs: 'cell_id_index', 'age', 'donor_id', 'sex', 'subtissue', 'cell_name', 'cell_id', 'cell_clean', 'specific_cell_type', 'medium_cell_type', 'seqtech', 'library_size', 'total_junction_reads', 'annotated_junction_reads', 'unannotated_junction_reads', 'n_detected_annotated_junctions', 'n_detected_unannotated_junctions', 'age_numeric', 'age_group', 'cell_ontology_class', 'broad_cell_type', 'tissue', 'tissue_celltype', 'dataset'
    var: 'index', 'junction_id', 'event_id', 'splice_motif', 'annotation_status', 'gene_name', 'gene_id', 'num_junctions', 'position_off_5_prime', 'position_off_3_prime', 'CountJuncs', 'junction_id_index', 'n_cells_detected', 'confidence', 'aging_gene', 'aging_lifespan_effect', 'aging_longevity_influence', 'RBP_gene', 'Aging_gene'
    obsm: 'X_PHI'
    varm: 'psi_learned'

### Add module labels to cell types 

In [ ]:
# Always assert after creation or reuse
assert all(splice_adata.obs_names == ge_adata.obs_names)
splice_adata.obs["total_counts"] = ge_adata.obs["total_counts"]

In [ ]:
# update broad_cell_type, medium_cell_type, tissue, tissue_celltype to be the same in ge_adata.obs as in splice_adata.obs
ge_adata.obs["broad_cell_type"] = splice_adata.obs["broad_cell_type"]
ge_adata.obs["medium_cell_type"] = splice_adata.obs["medium_cell_type"]
ge_adata.obs["tissue"] = splice_adata.obs["tissue"]
ge_adata.obs["tissue_celltype"] = splice_adata.obs["tissue_celltype"]
# Clean up Broad Cell Type column
ge_adata.obs["age_numeric"] = splice_adata.obs["age_numeric"]
splice_adata.obs["GAG_score"] = ge_adata.obs["GAG_score"]
assert np.all(ge_adata.obs_names == splice_adata.obs_names)

In [ ]:
# PLot violin chart of GAG_score vs age_numeric 
plt.figure(figsize=(10, 6))
sns.violinplot(data=ge_adata.obs, x="age_numeric", y="GAG_score", inner="box", scale="width")
plt.show()

## Rename factors to be in order that matches PI

In [ ]:
# Number of factors (K)
K = splice_adata.obsm["X_PHI"].shape[1]

# Get new factor order (indices sorted by decreasing PI)
new_order = np.argsort(-pi)  # minus for descending sort

# Print sorted pi with original indices
print("Sorted pi values with original factor indices:")
for rank, idx in enumerate(new_order):
    print(f"Rank {rank + 1}: Factor {idx} → PI = {pi[idx]:.4f}")

old_to_new_factor_idx = {old: new for new, old in enumerate(new_order)}
print(old_to_new_factor_idx)

# Reorder X_PHI (cell × K)
splice_adata.obsm["X_PHI"] = splice_adata.obsm["X_PHI"][:, new_order]

# Reorder psi_learned (junction × K)
splice_adata.varm["psi_learned"] = splice_adata.varm["psi_learned"][:, new_order]

# Generate column names: SP_1, SP_2, ..., SP_K
phi_colnames = [f"SP_{i+1}" for i in range(K)]

# Create a DataFrame from X_PHI
phi_df = pd.DataFrame(
    splice_adata.obsm["X_PHI"], 
    index=splice_adata.obs_names,
    columns=phi_colnames
)

# Merge with adata.obs
splice_adata.obs = pd.concat([splice_adata.obs, phi_df], axis=1)

# Update differential splicing results also 
# Rename and map
diff_spl = diff_spl.rename(columns={"factor_idx": "old_factor_idx"})
diff_spl["factor_idx"] = diff_spl["old_factor_idx"].map(old_to_new_factor_idx)

In [ ]:
splice_adata.obsm["X_PHI"]

In [ ]:
leaflet_model["assign_post"][:, 1]

In [ ]:
# --- Setup and Data Loading (Same as before) ---
psis_loc = leaflet_model["psis_loc"] # Shape: [K, J] -> [20, 70102]
psis_scale = leaflet_model["psis_scale"] # Shape: [K, J] -> [20, 70102]
print(f"First few values of psis_loc: {psis_loc[:5, :5]}")
print(f"First few values of psis_scale: {psis_scale[:5, :5]}")

# Reorder values in psis_loc and psis_scale according to new factor order
psis_loc = psis_loc[new_order, :]
psis_scale = psis_scale[new_order, :]
print(f"First few values of psis_loc after reordering: {psis_loc[:5, :5]}")
print(f"First few values of psis_scale after reordering: {psis_scale[:5, :5]}")

S = 100 # Number of samples (S)
K = psis_loc.shape[0] # Number of factors (K=20)
J = psis_loc.shape[1] # Number of junctions (J=70102)

print(f"Sampling Parameters: S={S}, K={K}, J={J}")

# --- 1. Full Factorial Sampling (Unconstrained Space) ---

# Expand dimensions of psis_loc and psis_scale: [K, J] -> [K, J, 1]
# This makes them broadcastable for the S=100 samples.
loc_expanded = psis_loc[:, np.newaxis, :] # Shape [20, 1, 70102]
scale_expanded = psis_scale[:, np.newaxis, :] # Shape [20, 1, 70102]

# Use np.random.normal to sample from ALL K*J distributions at once.
# The size argument ensures we get S samples for each (K, J) pair.
# Resulting shape: [K, S, J]
psi_samples_logit_K_S_J = np.random.normal(
    loc=loc_expanded, 
    scale=scale_expanded, 
    size=(K, S, J)
)

# --- 2. Convert to PSI (0-1 Range) ---

# Apply the inverse logit (Sigmoid) function
psi_samples_psi_K_S_J = 1 / (1 + np.exp(-psi_samples_logit_K_S_J))

# --- 3. Reorder to [S, K, J] and Finalize ---

# Transpose the array to the required shape [S, K, J]
psi_samples_psi_S_K_J = np.transpose(psi_samples_psi_K_S_J, (1, 0, 2))

# Convert to torch.Tensor as required by the compute_psi_effect_size function
psi_samples_torch = torch.from_numpy(psi_samples_psi_S_K_J).float()

# --- 4. Add to Model Dictionary ---
leaflet_model["psi_samples"] = psi_samples_torch

print(f"\n✅ Finished processing and sampling in [S, K, J] format.")
print(f"   Key 'psi_samples' added to leaflet_model.")
print(f"   Shape of final samples (torch.Tensor): {leaflet_model['psi_samples'].shape}")
print(f"   Range check (min/max): {np.min(psi_samples_psi_S_K_J):.4f} / {np.max(psi_samples_psi_S_K_J):.4f}")

In [ ]:
# Extract X_PHI matrix and convert to DataFrame
phi = pd.DataFrame(
    splice_adata.obsm["X_PHI"],
    index=splice_adata.obs["cell_id"]
)
phi.columns = [f"SP_{i+1}" for i in range(phi.shape[1])]

phi_long = phi.reset_index().melt(id_vars="cell_id", var_name="Splicing Program", value_name="Activity")

# Plot
plt.figure(figsize=(12, 6))  # Adjust width as needed
sns.violinplot(data=phi_long, x="Splicing Program", y="Activity", inner="box", scale="width")
plt.xticks(rotation=45)
plt.title("Distribution of Latent Splicing Program Activities Across Cells")
plt.xlabel("Latent Splicing Program (SP)")
plt.ylabel("Activity")
plt.tight_layout()
plt.show()

In [ ]:
# Setup
phi_matrix = splice_adata.obsm["X_PHI"]
factor_names = [f"SP_{i+1}" for i in range(phi_matrix.shape[1])]

In [ ]:
splice_adata.obs[["age_numeric", "age_group"]].value_counts()

### Calculate perplexity 

In [ ]:
PHI = splice_adata.obsm["X_PHI"]
# Calculate cell perplexity/entropy 
print("Calculating cell perplexity...")
PHI_safe = np.clip(PHI, 1e-10, 1)  # Prevent log(0) errors
entropy = -np.sum(PHI_safe * np.log(PHI_safe), axis=1)
perplexity = np.exp(entropy)
splice_adata.obs["perplexity"] = perplexity

In [ ]:
# Build DataFrame
phi_df = pd.DataFrame(phi_matrix, index=splice_adata.obs_names, columns=factor_names)
phi_df["broad_cell_type"] = splice_adata.obs["broad_cell_type"].values
phi_df["medium_cell_type"] = splice_adata.obs["medium_cell_type"].values
phi_df["tissue_celltype"] = splice_adata.obs["tissue_celltype"].values
phi_df["age_group"] = splice_adata.obs["age_group"].values
phi_df["sex"] = splice_adata.obs["sex"].values
phi_df["age_numeric"] = splice_adata.obs["age_numeric"].values
phi_df["perplexity"] = splice_adata.obs["perplexity"].values

In [ ]:
# save phi_df as Supplementary Table 2 as excel file 
phi_df.to_excel(os.path.join(OUTPUT_DIR, "Supplementary_Table_2.xlsx"), index=False)
print(f"Saved phi_df as Supplementary Table 2 to: {os.path.join(OUTPUT_DIR, 'Supplementary_Table_2.xlsx')}")

In [ ]:
# Calculate median perplexity value across all cells 
median_perplexity = phi_df["perplexity"].median()
print(f"Median perplexity value across all cells: {median_perplexity}")

## Visualize Pi vs PVE

In [ ]:
# get reordered pi values 
new_pi = pi[new_order]

# visualize global learned pi values, vs how much variance each one explains across cells PHI? 
# How much variance each factor contributes in the loadings
phi_var = np.var(splice_adata.obsm["X_PHI"], axis=0)  # Variance across cells for each factor
pve_factor_space = phi_var / np.sum(phi_var)
print(pve_factor_space)

In [ ]:
# Plot variance across cells for each factor 
plt.figure(figsize=(10, 6))
plt.bar(factor_names, phi_var)
plt.xlabel("Splicing Program (SP)")
plt.ylabel("Variance")
plt.title("Variance across cells for each factor")
# plot value on top of each bar 
for i, v in enumerate(phi_var):
    plt.text(i, v, f"{v:.2f}", ha='center', va='bottom')
# rotate x labels 90 degrees 
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Simple scatter plot
plt.figure(figsize=(5, 3))

# Use scatter with edgecolors for black outline and c parameter for color mapping
plt.scatter(new_pi, pve_factor_space, s=50, alpha=0.8, 
            c=new_pi, cmap='viridis', edgecolors='black', linewidth=1)

# Add colorbar to show π value mapping
plt.colorbar(label='π value')

# Add factor labels for interesting points
for i in range(len(new_pi)):
    if i < 5 or pve_factor_space[i] > 0.05:  # Label first 3 and high variance factors
        plt.annotate(f'SP{i+1}', (new_pi[i], pve_factor_space[i]), 
                    xytext=(2, 2), textcoords='offset points', fontsize=8)  # Increased from (5,5) to (10,10)

plt.xlabel('Global Importance (π)')
plt.ylabel('Percent Variance Explained (PVE)')
plt.title('Global vs Cell-level Factor Importance')
plt.xlim(0, 0.25)

# add dashed hori line at 0.01
plt.axhline(0.01, color='gray', linestyle='--', linewidth=1)
plt.axvline(0.01, color='gray', linestyle='--', linewidth=1)

# Add SP definition at bottom of plot
plt.text(0.15, 0.02, 'SP: Splicing Program', fontsize=8, style='italic', 
         transform=plt.gca().transData, verticalalignment='bottom')

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Assume new_pi and pve_factor_space already exist
num_factors = len(new_pi)
factor_labels = [f'SP{i+1}' for i in range(num_factors)]

# Normalize PVE for colormap
norm = plt.Normalize(pve_factor_space.min(), pve_factor_space.max())
colors = plt.cm.viridis(norm(pve_factor_space))

# Set up figure and axis
fig, ax = plt.subplots(figsize=(5, 3))

# Bar plot
bars = ax.bar(
    factor_labels,
    new_pi,
    color=colors,
    edgecolor='black'
)

# Create scalar mappable and colorbar
sm = plt.cm.ScalarMappable(cmap="viridis", norm=norm)
sm.set_array([])  # Required dummy
cbar = plt.colorbar(sm, ax=ax)  # Pass ax explicitly
cbar.set_label("% Variance Explained (PVE)")

# Add horizontal threshold line
ax.axhline(0.01, color='gray', linestyle='--', linewidth=1)

# Formatting
ax.set_ylabel('Global Importance (π)')
ax.set_xlabel('Splicing Programs (SP)')
# ax.set_title('Factor Importance (π), colored by PVE')
ax.set_xticks(np.arange(num_factors))
ax.set_xticklabels(factor_labels, rotation=90, fontsize=10)

plt.tight_layout()

# Save to output_dir with today's date OUTPUT_DIR 
output_path = os.path.join(OUTPUT_DIR, f"factor_pi_vs_pve.pdf")
plt.savefig(output_path, format='pdf', bbox_inches='tight')
print(f"Plot saved to: {output_path}")

plt.show()


In [ ]:
# What's the total PVE from 1-5? 
pve_factor_space[0:5].sum()

In [ ]:
# save splice_data for figure making 
output_dir="/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Mouse_Splicing_Foundation/Figures"

In [ ]:
# Good balance: faster than gzip, smaller than no compression
splice_adata.write_h5ad(
    os.path.join(output_dir, "splice_adata_for_figures_mouse_foundation.h5ad"),
    compression="lzf"
)
print(f"Saved splice_adata for figures to: {os.path.join(output_dir, 'splice_adata_for_figures_mouse_foundation.h5ad')}")

In [ ]:
#splice_adata.layers["denoised_PSI"] = splice_adata.obsm["X_PHI"] @ splice_adata.varm["psi_learned"].T
#print(f"The shape of denoised_PSI is: {splice_adata.layers['denoised_PSI'].shape}")

In [ ]:
splice_adata

In [ ]:
# remove all the layers from ge_adata .layers["length_norm", "log_norm", "raw_counts"]
ge_adata.layers.pop("length_norm", None)
ge_adata.layers.pop("log_norm", None)
ge_adata.layers.pop("raw_counts", None)

# remove .obsp["connectivities"], .obsp["distances"]
ge_adata.obsp.pop("connectivities", None)
ge_adata.obsp.pop("distances", None)

In [ ]:
# Save ge_adata for figure making 
ge_adata.write_h5ad(
    os.path.join(output_dir, "ge_adata_for_figures_mouse_foundation.h5ad"),
    compression="lzf"
)
print(f"Saved ge_adata for figures to: {os.path.join(output_dir, 'ge_adata_for_figures_mouse_foundation.h5ad')}")

In [ ]:
# Assert order of cells is the same
assert np.all(ge_adata.obs_names == splice_adata.obs_names)

In [ ]:
# directory for saving plots
OUTPUT_DIR = "/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Mouse_Splicing_Foundation/Figures/FIGURE3/figures"
# make plots dir if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Saving plots to: {OUTPUT_DIR}")

In [ ]:
plt.figure(figsize=(3, 3))

# Boxplot: keep as vector
sns.boxplot(
    data=splice_adata.obs,
    x="age_numeric", 
    y="perplexity", 
    color="lightgray", 
    fliersize=1, 
    linewidth=0.8
)

# Stripplot: will rasterize below
stripplot = sns.stripplot(
    data=splice_adata.obs,
    x="age_numeric", 
    y="perplexity", 
    color="darkslategrey", 
    size=0.5, 
    jitter=False,
    alpha=0.3
)

# 🟡 Rasterize only the point cloud layer
for c in stripplot.collections:
    c.set_rasterized(True)

# Median labels
medians = (
    splice_adata.obs
    .groupby("age_numeric", observed=True)["perplexity"]
    .median()
)
for i, (age, median_val) in enumerate(medians.items()):
    plt.text(
        x=i,
        y=median_val + 0.5,
        s=f"{median_val:.2f}",
        ha='center',
        va='bottom',
        fontsize=7,
        color='black'
    )

# Formatting
plt.xlabel("Age (months)", fontsize=10)
plt.ylabel("# of Active Factors (Perplexity)", fontsize=10)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

for spine in plt.gca().spines.values():
    spine.set_linewidth(0.5)

plt.tight_layout()

# Save to PDF with rasterized points
output_path = f"{OUTPUT_DIR}/perplexity_by_age_rasterized.pdf"
plt.savefig(output_path, bbox_inches='tight', dpi=300)
print(f"Saved plot to: {output_path}")
plt.show()

In [ ]:
plt.figure(figsize=(3, 3))

# Boxplot: keep as vector
sns.boxplot(
    data=splice_adata.obs,
    x="sex", 
    y="perplexity", 
    color="lightgray", 
    fliersize=1, 
    linewidth=0.8
)

# Stripplot: will rasterize below
stripplot = sns.stripplot(
    data=splice_adata.obs,
    x="sex", 
    y="perplexity", 
    color="darkslategrey", 
    size=0.5, 
    jitter=False,
    alpha=0.3
)

# 🟡 Rasterize only the point cloud layer
for c in stripplot.collections:
    c.set_rasterized(True)

# Median labels
medians = (
    splice_adata.obs
    .groupby("sex", observed=True)["perplexity"]
    .median()
)
for i, (age, median_val) in enumerate(medians.items()):
    plt.text(
        x=i,
        y=median_val + 0.5,
        s=f"{median_val:.2f}",
        ha='center',
        va='bottom',
        fontsize=7,
        color='black'
    )

# Formatting
plt.xlabel("Sex", fontsize=10)
plt.ylabel("# of Active Factors (Perplexity)", fontsize=10)
plt.xticks(fontsize=10)
plt.yticks(fontsize=10)

for spine in plt.gca().spines.values():
    spine.set_linewidth(0.5)

plt.tight_layout()

# Save to PDF with rasterized points
output_path = f"{OUTPUT_DIR}/perplexity_by_sex_rasterized.pdf"
plt.savefig(output_path, bbox_inches='tight', dpi=300)
print(f"Saved plot to: {output_path}")
plt.show()

In [ ]:
from scipy.stats import mannwhitneyu, ttest_ind

# Split by sex
grouped = splice_adata.obs.groupby("sex", observed=True)["perplexity"]
groups = [vals.values for _, vals in grouped]

# Mann–Whitney U test
u_stat, p_val = mannwhitneyu(groups[0], groups[1], alternative="two-sided")
print(f"Mann–Whitney U: U={u_stat:.2f}, p={p_val:.3e}")

# (Optional) parametric t-test
t_stat, p_val_t = ttest_ind(groups[0], groups[1], equal_var=False)
print(f"T-test: t={t_stat:.2f}, p={p_val_t:.3e}")

In [ ]:
# --- The Complete, Corrected Plotting Script ---

## 🛠️ Step 1: Data Preparation

# Only consider tissue_celltype groups with >= 100 cells for plotting
celltype_counts = splice_adata.obs["tissue_celltype"].value_counts()
valid_celltypes = celltype_counts[celltype_counts >= 100].index
plot_obs = splice_adata.obs[splice_adata.obs["tissue_celltype"].isin(valid_celltypes)].copy()

# Compute median perplexity by tissue_celltype and get top 20
median_order = (
    plot_obs
    .groupby("tissue_celltype", observed=True)["perplexity"]
    .median()
    .sort_values(ascending=False)
    .head(50)
    .index
)

# Color palette for broad_cell_type
unique_broad = plot_obs["broad_cell_type"].unique()
palette = sns.color_palette("tab20", len(unique_broad))
palette_dict = dict(zip(unique_broad, palette))

# Map the tissue_celltype to its broad_cell_type color
color_map = plot_obs.set_index("tissue_celltype")["broad_cell_type"].to_dict()

# Create a list of colors in the exact order of the boxplots (median_order)
plot_colors = [palette_dict[color_map[cell_type]] for cell_type in median_order]


## 📊 Step 2: Main Plotting

# Start figure
fig, ax = plt.subplots(figsize=(8, 6))

# Main plot - REMOVE 'hue' and use 'plot_colors' to center the boxplots
bp = sns.boxplot(
    data=plot_obs,
    y="tissue_celltype", 
    x="perplexity", 
    fliersize=0.6, 
    linewidth=0.8,
    order=median_order,
    palette=plot_colors, # Use the pre-calculated list of colors
    ax=ax
)

# Rasterize fliers (optional, for PDF size optimization)
for c in ax.collections:
    if isinstance(c, plt.collections.PathCollection):
        c.set_rasterized(True)

# Global median line
global_median = splice_adata.obs["perplexity"].median()
ax.axvline(global_median, color='gray', linestyle='--', linewidth=1)
ax.text(global_median + 0.3, -1, f"Global Median: {global_median:.2f}",
        color='gray', fontsize=6, va='top')


## 🎨 Step 3: Legend and Formatting

# Manually create the legend since 'hue' was removed
# 1. Get unique broad cell types present in the top 20
broad_types_in_plot = sorted(list(set(color_map[cell] for cell in median_order)))

# 2. Create legend handles and labels
handles = [plt.Rectangle((0, 0), 1, 1, fc=palette_dict[bt]) for bt in broad_types_in_plot]
labels = broad_types_in_plot

ax.legend(
    handles=handles,
    labels=labels,
    title="Cell Type Category",
    bbox_to_anchor=(1.3, 1),
    loc='upper left',
    fontsize=8,
    title_fontsize=8,
    frameon=True
)

# Axis formatting
ax.set_ylabel("Cell Type", fontsize=10)
ax.set_xlabel("# of Active Splicing Programs (Perplexity)", fontsize=10)
ax.tick_params(axis='y', labelsize=8)

# Border
for spine in ax.spines.values():
    spine.set_linewidth(0.5)

# Save
plt.tight_layout()
pdf_path = f"{OUTPUT_DIR}/perplexity_by_cell_type_ordered_rasterized.pdf"
plt.savefig(pdf_path, bbox_inches='tight', dpi=300)
print(f"Saved corrected plot to: {pdf_path}")
plt.show()

In [ ]:
median_order

In [ ]:
# 1. Reuse your filtering logic
celltype_counts = splice_adata.obs["tissue_celltype"].value_counts()
valid_celltypes = celltype_counts[celltype_counts >= 0].index
df_clean = splice_adata.obs[splice_adata.obs["tissue_celltype"].isin(valid_celltypes)].copy()

# 2. Quick Descriptive Summary (The "Eye Test")
# This tells you the raw rankings immediately
summary_stats = df_clean.groupby("broad_cell_type")["perplexity"].agg(['mean', 'median', 'count']).sort_values('median', ascending=False)
print("--- Ranking by Median Perplexity ---")
print(summary_stats)

# 3. Linear Regression (The Statistical Test)
# Formula: Predict perplexity based on broad_cell_type categories
# Note: Statsmodels automatically handles the categorical conversion
model = smf.ols("perplexity ~ C(broad_cell_type)", data=df_clean).fit()

# 4. Extract and print significant results
# The coefficients show how much higher/lower a group is compared to the 'Reference' group
print("\n--- Regression Results ---")
# This prints the full statistical table
print(model.summary()) 

# Optional: Cleaner view of just the coefficients
results_df = pd.DataFrame({
    'coef': model.params,
    'p_value': model.pvalues
}).sort_values('coef', ascending=False)

print("\n--- Coefficients (sorted) ---")
print(results_df)

## Get correlation matrix plot

In [ ]:
# Remake PHI matrix correlation plot
X_PHI_corr_matrix = plot_correlation_matrix(splice_adata.obsm["X_PHI"], OUTPUT_DIR)

## Quick scVI check

In [ ]:
np.all(splice_adata.obs_names == ge_adata.obs_names)

## More detailed aging analysis

### Figure 2 Factor Covaraites Labels

In [ ]:
def run_variance_explained_analysis(
    splice_adata, 
    sample_id,
    cell_type_col,
    PLOTS_DIR=None, 
    DATA_DIR=None,
    f_display_mode="log10",       # "raw", "log10", "zscore_by_term"
    f_clip_percentile=99,         # upper clip for F visualization
    plot_prop=True,               # plot PropSS heatmap
    plot_F=True                   # plot F-stat heatmap
):
    """
    Variance attribution for splicing programs (LeafletFA factors).

    Steps:
      1) Z-score factors (columns) to remove scale bias.
      2) OLS per factor with Type-II ANOVA:
           SP_k ~ C(cell_type) + C(tissue) + C(sex) + age + C(dataset)
                   + C(cell_type):age + C(sex):age
      3) Plot:
         (A) Proportion of explained variance (sum_sq share) per factor×term
         (B) F-statistic heatmap, with display modes:
              - "raw": clipped raw F
              - "log10": log10(F+1) then clip
              - "zscore_by_term": column-wise z-score of F
         Both use identical row order (from PropSS clustering if available).
    Returns:
      dict with raw/processed tables and row order used for plotting.
    """
    print("Running variance explained analysis...")

    # ---------- Build analysis frame with z-scored factors ----------
    X_phi = splice_adata.obsm["X_PHI"]
    X_phi_z = StandardScaler(with_mean=True, with_std=True).fit_transform(X_phi)
    factor_names = [f"SP_{i+1}" for i in range(X_phi_z.shape[1])]
    analysis_df = pd.DataFrame(X_phi_z, index=splice_adata.obs_names, columns=factor_names)

    # Covariates
    obs_cols_to_copy = {
        cell_type_col: "cell_type",
        "tissue": "tissue",
        "sex": "sex",
        sample_id: "dataset",
        "age_numeric": "age",
    }
    for obs_col, df_col in obs_cols_to_copy.items():
        analysis_df[df_col] = splice_adata.obs[obs_col].values

    # ---------- Model terms ----------
    base_terms = ['C(cell_type)', 'C(tissue)', 'C(sex)', 'age', 'C(dataset)']
    interaction_terms = ['C(cell_type):age', 'C(sex):age']
    all_terms = base_terms + interaction_terms

    # ---------- Fit OLS + ANOVA per factor ----------
    r2_records, anova_tables = [], []
    for factor_col in tqdm(factor_names, desc="Analyzing factors for variance explained"):
        formula = f"{factor_col} ~ " + " + ".join(all_terms)
        model = smf.ols(formula, data=analysis_df, missing='drop').fit()
        if model.nobs < (model.df_model + 2) or model.df_resid <= 0:
            print(f"Skipping {factor_col}: insufficient observations")
            continue
        r2_records.append({'SP': factor_col, 'r2_overall': model.rsquared, 'n_obs': int(model.nobs)})
        a2 = anova_lm(model, typ=2)
        a2['SP'] = factor_col
        anova_tables.append(a2.reset_index())

    r2_df = pd.DataFrame(r2_records)
    anova_df = pd.concat(anova_tables, ignore_index=True)

    # ---------- Clean & ensure numeric ----------
    anova_df = anova_df.rename(columns={'index': 'covariate_term'})
    anova_df = anova_df[anova_df['covariate_term'] != 'Residual'].copy()
    for col in ['sum_sq', 'F']:
        anova_df[col] = pd.to_numeric(anova_df[col], errors='coerce')

    # ---------- (A) Proportion of explained variance (PropSS) ----------
    ss_pivot = anova_df.pivot_table(index='SP', columns='covariate_term', values='sum_sq', aggfunc='sum')
    ss_pivot = ss_pivot.loc[:, (ss_pivot.sum(axis=0).abs() > 1e-12)]
    total_ss = ss_pivot.sum(axis=1)
    prop_var = ss_pivot.div(total_ss.replace(0, np.nan), axis=0)

    # Row labels with R^2
    r2_map = (
        r2_df.set_index('SP')
            .loc[prop_var.index]
            .assign(row=lambda d: [f"{sp} (R²={r2:.2f})" for sp, r2 in zip(d.index, d['r2_overall'])])
            ['row'].to_dict()
    )
    prop_var_disp = prop_var.rename(index=r2_map).astype(float)

    # ---------- (B) F-statistic processing ----------
    f_pivot = anova_df.pivot_table(index='SP', columns='covariate_term', values='F', aggfunc='mean')
    f_pivot = f_pivot.reindex(prop_var.index)  # align to PropSS row order

    F_vis = f_pivot.replace([np.inf, -np.inf], np.nan).copy()
    cbar_label = ""
    cmap = "viridis"; vmin = None; vmax = None

    if f_display_mode == "log10":
        F_vis = np.log10(F_vis + 1.0)
        cbar_label = "log₁₀(F + 1)"
        clip_hi = np.nanpercentile(F_vis.values.ravel(), f_clip_percentile) if np.isfinite(F_vis.values).any() else None
        if clip_hi and clip_hi > 0:
            F_vis = F_vis.clip(upper=clip_hi)
        vmin, vmax = 0, clip_hi
    elif f_display_mode == "zscore_by_term":
        # column-wise z-score
        F_vis = (F_vis - F_vis.mean(axis=0)) / F_vis.std(axis=0)
        cbar_label = "F (z-score within term)"
        clip_hi = np.nanpercentile(F_vis.values.ravel(), f_clip_percentile)
        clip_lo = np.nanpercentile(F_vis.values.ravel(), 100 - f_clip_percentile)
        F_vis = F_vis.clip(lower=clip_lo, upper=clip_hi)
        cmap = "PRGn"
        vmin, vmax = None, None
    else:  # "raw"
        cbar_label = "F-statistic (clipped)"
        clip_hi = np.nanpercentile(F_vis.values.ravel(), f_clip_percentile) if np.isfinite(F_vis.values).any() else None
        if clip_hi and clip_hi > 0:
            F_vis = F_vis.clip(upper=clip_hi)
        cmap = "viridis"
        vmin, vmax = 0, clip_hi

    F_disp = F_vis.rename(index=r2_map).astype(float)

    # ---------- Plot A: PropSS ----------
    cg_prop = None
    if plot_prop:
        width, height = 4.0, 5.0
        annot_prop = prop_var_disp.round(2).astype(str)
        cg_prop = sns.clustermap(
            prop_var_disp, cmap="PRGn", center=0,
            linewidths=0.2, linecolor='black',
            annot=annot_prop, fmt='',
            annot_kws={"size": 4, "color": "grey"},
            figsize=(width, height),
            xticklabels=True, yticklabels=True,
            vmin=0, vmax=1.0
        )
        plt.setp(cg_prop.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=8)
        plt.setp(cg_prop.ax_heatmap.get_yticklabels(), rotation=0, fontsize=8)
        cg_prop.ax_heatmap.set_xlabel('Covariate Terms', fontsize=11, fontweight='bold')
        cg_prop.ax_heatmap.set_ylabel('LeafletFA Splicing Programs', fontsize=11, fontweight='bold')
        if PLOTS_DIR:
            out_prop = os.path.join(PLOTS_DIR, f"variance_explained_propSS_{cell_type_col}.pdf")
            cg_prop.savefig(out_prop, format="pdf", bbox_inches="tight")
            print(f"  Saved: {out_prop}")

    # ---------- Plot B: F-stat ----------
    if plot_F:
        # keep same row order as PropSS if we clustered it
        if cg_prop is not None and cg_prop.dendrogram_row is not None:
            row_idx = cg_prop.dendrogram_row.reordered_ind
            F_disp_ordered = F_disp.iloc[row_idx]
        else:
            F_disp_ordered = F_disp

        # sparse annotations: only show top 5% values for readability (for log/raw modes)
        if f_display_mode in ("raw", "log10"):
            thresh = np.nanpercentile(F_disp_ordered.values.ravel(), 95)
            annot_F = F_disp_ordered.where(F_disp_ordered >= thresh).round(2).astype(str)
        else:
            annot_F = F_disp_ordered.round(2).astype(str)

        cg_F = sns.clustermap(
            F_disp_ordered,
            cmap=cmap,
            linewidths=0.2, linecolor='black',
            annot=annot_F, fmt='',
            annot_kws={"size": 4},
            figsize=(5, 4),
            xticklabels=True, yticklabels=True,
            col_cluster=True, row_cluster=(cg_prop is None),
            vmin=vmin, vmax=vmax
        )
        plt.setp(cg_F.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=8)
        plt.setp(cg_F.ax_heatmap.get_yticklabels(), rotation=0, fontsize=8)
        cg_F.ax_heatmap.set_xlabel('Covariate Terms', fontsize=11, fontweight='bold')
        cg_F.ax_heatmap.set_ylabel('LeafletFA Splicing Programs', fontsize=11, fontweight='bold')
        cbar = cg_F.ax_heatmap.collections[0].colorbar
        cbar.ax.tick_params(labelsize=8)
        cbar.set_label(cbar_label, fontsize=9)
        if PLOTS_DIR:
            out_F = os.path.join(PLOTS_DIR, f"variance_explained_Fstat_{f_display_mode}_{cell_type_col}.pdf")
            cg_F.savefig(out_F, format="pdf", bbox_inches="tight")
            print(f"  Saved: {out_F}")

    plt.show()

    return {
        "anova_raw": anova_df,           # long-form ANOVA table (no residual)
        "prop_var": prop_var,            # proportions per factor × term
        "F_raw": f_pivot,                # raw (unclipped) F values
        "prop_var_display": prop_var_disp,
        "F_display": F_disp,             # processed for visualization
        "r2": r2_df
    }

In [ ]:
phi_df

In [ ]:
# read in factor labels 
factor_labels_medium_cell_type = run_variance_explained_analysis(splice_adata, cell_type_col="tissue_celltype", sample_id="dataset", PLOTS_DIR=OUTPUT_DIR)

### Subset aging dataset

In [ ]:
# subset to cells with age > 2 and at least 100 young and 100 old cells
splice_adata_full = splice_adata.copy()
ge_adata_full = ge_adata.copy()
ge_adata = ge_adata[splice_adata.obs_names].copy()
assert np.all(ge_adata.obs_names == splice_adata.obs_names)

In [ ]:
# ── 1.  Data → DataFrame ───────────────────────────────────────────────────────
ψ  = splice_adata.varm["psi_learned"]          # (J, K)
K  = ψ.shape[1]

# Subtract row (junction) mean from each factor value
ψ_centered = ψ - np.nanmean(ψ, axis=1, keepdims=True)
K = ψ_centered.shape[1]
df = pd.DataFrame(ψ_centered, columns=[f"SP{j+1}" for j in range(K)])

# ── 2.  ρ and p-values for *each* pair ─────────────────────────────────────────
rho    = np.zeros((K, K))
pvals  = np.ones((K, K))

for i in range(K):
    for j in range(i+1, K):
        r, p = spearmanr(df.iloc[:, i], df.iloc[:, j], nan_policy='omit')
        rho[i, j] = rho[j, i] = r
        pvals[i, j] = pvals[j, i] = p

# ── 3.  Benjamini–Hochberg FDR adjustment over the upper triangle ─────────────
tri_idx         = np.triu_indices(K, k=1)          # exclude the diagonal
p_flat          = pvals[tri_idx]
_, q_flat, _, _ = multipletests(p_flat, method="fdr_bh")
qvals           = np.ones_like(pvals)
qvals[tri_idx]  = qvals.T[tri_idx] = q_flat

# ── 4.  Significance mask  (FDR < 0.05 and ρ > 0.20) ──────────────────────────
star_mask = (qvals < 0.05) & (rho > 0.20)
annot     = np.where(star_mask, "*", "")

# ── 5.  Build DataFrames with indices/columns for seaborn ─────────────────────
rho_df   = pd.DataFrame(rho,   index=df.columns, columns=df.columns)
annot_df = pd.DataFrame(annot, index=df.columns, columns=df.columns)

# ── 1.  clustered heat-map with all tick labels ───────────────────────────────
sns.set_theme(style="white")
g = sns.clustermap(
    rho_df,
    cmap="vlag", center=0, linewidths=0.4, square=True,
    metric="euclidean", row_cluster=True, col_cluster=True,
    figsize=(6, 6),
    annot=annot_df, fmt="", annot_kws={"size":7},
    xticklabels=True, yticklabels=True
)

# Rotate x-labels so they don’t overlap
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90)
plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0)

g.ax_heatmap.set_title("Spearman ρ between ψ-factors\n*  FDR < 0.05 & ρ > 0.20  *", pad=14)
g.ax_heatmap.set_xlabel("Factors")
g.ax_heatmap.set_ylabel("Factors")

plt.tight_layout()

# ── 2.  save to disk ──────────────────────────────────────────────────────────
g.savefig(f"{OUTPUT_DIR}/psi_factor_spearman_clustermap.pdf", bbox_inches="tight")

# ── 3.  report extreme pairs ──────────────────────────────────────────────────
# flatten upper triangle (exclude diagonal), keep finite values only
tri_i, tri_j = np.triu_indices_from(rho_df, k=1)
tri_vals = rho_df.values[tri_i, tri_j]

# build a DataFrame for easy sorting
pairs_df = pd.DataFrame({
    "factor_i": rho_df.index[tri_i],
    "factor_j": rho_df.columns[tri_j],
    "rho":      tri_vals
}).dropna()

# top 2 positive
top_pos = pairs_df.nlargest(2, "rho")
# top 2 negative
top_neg = pairs_df.nsmallest(2, "rho")
# top 2 least correlated (closest to zero)
top_flat = pairs_df.iloc[(pairs_df["rho"].abs()).nsmallest(2).index]

print("\nTop 2 positively correlated factor pairs:")
print(top_pos.to_string(index=False))

print("\nTop 2 negatively correlated factor pairs:")
print(top_neg.to_string(index=False))

print("\nTop 2 least correlated factor pairs:")
print(top_flat.to_string(index=False))


In [ ]:
plot_factor_violin(
    adata=splice_adata_full[splice_adata_full.obs["age_numeric"] > 2],
    factor_list=["SP_1", "SP_2", "SP_4", "SP_15", "SP_18"],
    groupby="age_numeric",  
    save_prefix="factor_age_violin"
)

In [ ]:
plot_factor_violin_sorted(
    adata=splice_adata,
    factor_list=["SP_3"],
    groupby="tissue_celltype",
    top_n=20,
    cmap="YlGnBu",
    save_prefix="sorted_violin",
    width=8,
    height=7
)

### Get heatmap for common cell types 

In [ ]:
# Get cell type counts by dataset to get common cell types
cell_type_counts = splice_adata_full.obs.groupby(['broad_cell_type', 'dataset']).size().unstack()
common_cell_types = cell_type_counts[(cell_type_counts["AB"] > 0) & (cell_type_counts["TMS"] > 0)]
common_cell_type_names = common_cell_types.index.tolist()

# Subset to common cell types
adata_common = splice_adata_full[
    splice_adata_full.obs["broad_cell_type"].isin(common_cell_type_names)
].copy()

# Convert to categorical first
adata_common.obs["broad_cell_type"] = adata_common.obs["broad_cell_type"].astype("category")
adata_common.obs["broad_cell_type"] = adata_common.obs["broad_cell_type"].cat.remove_unused_categories()

# Extract factor matrix and build DataFrame
num_factors = adata_common.obsm["X_PHI"].shape[1]
factor_cols = [f"SP_{i+1}" for i in range(num_factors)]

df = adata_common.obs[["broad_cell_type", "dataset"]].copy()
for i, col in enumerate(factor_cols):
    df[col] = adata_common.obsm["X_PHI"][:, i]

common_cell_type_names

In [ ]:
# Compute median factor values by cell type and dataset
median_profiles = (
    df.groupby(["broad_cell_type", "dataset"], observed=True)[factor_cols]
    .median()
    .reset_index()
)

# Create composite labels
median_profiles["label"] = (
    median_profiles["broad_cell_type"].astype(str)
    + " (" +
    median_profiles["dataset"].astype(str)
    + ")"
)

# Set label as index
median_profiles = median_profiles.set_index("label")[factor_cols]

# Plot clustermap (make shorter and add y-axis label "Broad Cell Types")
g = sns.clustermap(
    median_profiles,
    method="average",
    metric="euclidean",
    cmap="PRGn",
    yticklabels=True,
    center=0,
    linewidths=0.2,
    linecolor='gray',
    figsize=(5, 3)  # Shorter width, keep height
)

# Add x-axis and y-axis labels
g.ax_heatmap.set_xlabel("Splicing Programs", fontsize=10)
g.ax_heatmap.xaxis.label.set_size(10)
g.ax_heatmap.set_ylabel("Broad Cell Types", fontsize=10)
g.ax_heatmap.yaxis.label.set_size(10)

# Make sure all x-axis tick marks are shown
g.ax_heatmap.set_xticks(np.arange(len(median_profiles.columns)))  # one tick per factor
g.ax_heatmap.set_xticklabels(median_profiles.columns, rotation=90, fontsize=8)

# Format y-axis (cell types)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=8)

# Colorbar label and tick font
g.ax_cbar.set_label("Median Factor Activity")
g.ax_cbar.yaxis.label.set_size(7)
g.ax_cbar.tick_params(labelsize=6)

# Save
plt.tight_layout()
g.savefig(f"{OUTPUT_DIR}/clustering_common_celltype_dataset_factors.pdf", bbox_inches='tight')
print(f"Saved plot to {OUTPUT_DIR}/clustering_common_celltype_dataset_factors.pdf")
plt.show()

In [ ]:
splice_adata.obs["broad_cell_type"] = splice_adata.obs["broad_cell_type"].astype("category")
splice_adata.obs["medium_cell_type"] = splice_adata.obs["medium_cell_type"].astype("category")

In [ ]:
splice_adata.varm["psi_learned"] 

# Get junction usage across factors of junction 29467 index 
junc_psi = splice_adata.varm["psi_learned"][27224, :]
# convert to dataframe and add "SP_1" to SPN... and make barplot
junc_psi_df = pd.DataFrame(junc_psi)
# Rename main column to PSI 
junc_psi_df.columns = ["PSI"]
junc_psi_df["factor"] = [f"SP_{i+1}" for i in range(len(junc_psi_df))]
sns.barplot(x="factor", y="PSI", data=junc_psi_df)
# rotate x-axis labels
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Sanity check the diff_spl results, choose some of the top junctions and plot
# their learned PSI values across factors 
diff_spl[diff_spl["factor_idx"] == 3].sort_values(by="effect_size", ascending=True).head(10)

## Look at sig junctions from differential splicing analysis

In [ ]:
# Run the analysis
print("="*60)
print("TOP JUNCTIONS PER FACTOR PSI ANALYSIS")
print("="*60)

# Step 1: Get top junctions per factor
top_junctions_dict, subset_splice_adata, junction_factor_map = get_top_junctions_per_factor(
    diff_spl, 
    splice_adata, 
    top_n=10, 
    effect_size_col='abs_effect_size'
)

# Get unique junction indices
unique_junctions = junction_factor_map['junction_idx'].unique()

# Extract PSI learned values for these junctions
psi_subset = splice_adata.varm['psi_learned'][unique_junctions, :]

In [ ]:
# 1. Map row labels: get junction_id or gene_name from .var
# Get the integer indices into .varm (which uses original order)
unique_junctions = junction_factor_map['junction_idx'].unique()
# psi_learned: shape (n_junctions, n_factors), so index using integer indices
psi_subset = splice_adata.varm['psi_learned'][unique_junctions, :]
junction_id = splice_adata.var[splice_adata.var["junction_id_index"].isin(unique_junctions)]["junction_id"].values
gene_name = splice_adata.var[splice_adata.var["junction_id_index"].isin(unique_junctions)]["gene_name"].values

# Get number of factors (rows after transpose)
num_factors = psi_subset.shape[1]

# Create factor labels: F1, F2, ..., FK
factor_labels = [f"SP{i+1}" for i in range(num_factors)]

# 2. Create clustermap with color tweaks
g = sns.clustermap(
    psi_subset.T,
    center=0,
    figsize=(6, 5),
    xticklabels=False,
    yticklabels=factor_labels,
    cbar_kws={'label': 'PSI Loading'},
    robust=True  # clip extreme values for better contrast
)

# 3. Style adjustments
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=90, fontsize=8)
plt.setp(g.ax_heatmap.get_yticklabels(), rotation=0, fontsize=12)
g.ax_heatmap.set_ylabel("LeafletFA Factors")
g.ax_heatmap.set_xlabel("Highly DS Junctions")
plt.subplots_adjust(left=0.3, bottom=0.05, top=0.95, right=0.98)
save_path = os.path.join(OUTPUT_DIR, "psi_clustermap_top_10.pdf")
g.savefig(save_path, bbox_inches="tight")
print(f"Saved plot to {save_path}")
plt.show()

## Get UMAP on X_PHI

In [ ]:
ge_adata.layers["scVI_linear"] = ge_adata.obsm["X_normalized_scVI_linear"]

In [ ]:
# 1. Ensure the observation names (cell IDs) are perfectly matched
assert ge_adata.obs_names.equals(splice_adata.obs_names)

# 2. Get a list of ALL columns in splice_adata.obs that start with 'SP_'
sp_columns_to_transfer = splice_adata.obs.filter(regex='^SP_').columns

# 3. Transfer all identified columns to ge_adata.obs in one go
ge_adata.obs[sp_columns_to_transfer] = splice_adata.obs[sp_columns_to_transfer]

# 4. (Optional) Verify the transfer by printing the new columns in ge_adata.obs
print("Successfully transferred columns:")
print(ge_adata.obs.filter(regex='^SP_').columns.tolist())

In [ ]:
# Print all columns in .obs that start with 'SP_'
print(ge_adata.obs.filter(regex='^SP_').columns.tolist())

In [ ]:
# 1. Get the automatically generated list of 'SP_' columns from ge_adata.obs
splicing_programs = ge_adata.obs.filter(regex='^SP_').columns.tolist()

# 2. Define the other non-SP columns you always want to plot
other_metadata = ['broad_cell_type', 'age', "tissue", "mouse.id"]

# 3. Combine them into the final list for plotting
color_list = other_metadata + splicing_programs

import matplotlib.pyplot as plt
import os

# 4. For each color, generate a UMAP and save a rasterized PDF in OUTPUT_DIR
for color in color_list:
    sc.pl.umap(
        ge_adata, 
        color=color,
        ncols=1,
        frameon=False,
        show=False)
    fname = os.path.join(OUTPUT_DIR, f"umap_{color}.pdf")
    plt.savefig(fname, bbox_inches='tight', dpi=300)
    plt.close()
    print(f"Saved UMAP for '{color}' to {fname}")

In [ ]:
aging

### New circle heatmap combining delta age and activity across cell types

In [ ]:
# import Rectangle, Circle, and TextArea from matplotlib.patches
from matplotlib.patches import Rectangle

def plot_factor_circles(adata, cell_type_col='broad_cell_type', age_col='age_group'):
    """
    Clustered tile plot: background color shows delta PSI (old - young),
    circle outlines show activity (median nonzero expression).
    """
    # Extract factor matrix
    X_PHI = adata.obsm["X_PHI"]
    n_factors = X_PHI.shape[1]
    factor_cols = [f"SP_{i+1}" for i in range(n_factors)]
    factor_df = pd.DataFrame(X_PHI, index=adata.obs.index, columns=factor_cols)

    # Median expression (non-zero) per cell type
    median_expr = pd.DataFrame(index=adata.obs[cell_type_col].unique(), columns=factor_cols)
    for ct in median_expr.index:
        ct_mask = adata.obs[cell_type_col] == ct
        ct_data = factor_df.loc[ct_mask]
        for factor in factor_cols:
            vals = ct_data[factor]
            nonzero = vals[vals > 0]
            median_expr.loc[ct, factor] = nonzero.median() if len(nonzero) > 0 else 0
    median_expr = median_expr.astype(float).fillna(0)

    # Delta PSI: old - young
    delta_age = pd.DataFrame(index=median_expr.index, columns=factor_cols)
    for ct in median_expr.index:
        ct_mask = adata.obs[cell_type_col] == ct
        old_mask = ct_mask & (adata.obs[age_col] == 'old')
        young_mask = ct_mask & (adata.obs[age_col] == 'young')
        if old_mask.sum() > 10 and young_mask.sum() > 10:
            for factor in factor_cols:
                old_vals = factor_df.loc[old_mask, factor]
                young_vals = factor_df.loc[young_mask, factor]
                delta_age.loc[ct, factor] = old_vals.median() - young_vals.median()
        else:
            delta_age.loc[ct] = 0
    delta_age = delta_age.astype(float).fillna(0)

    # Clustering
    factor_order = list(range(len(factor_cols)))  # preserve original SP_1 ... SP_K order
    celltype_order = dendrogram(linkage(median_expr, method='ward'), no_plot=True)['leaves']
    celltype_order = [median_expr.index[i] for i in celltype_order]

    # Compute cell counts
    celltype_counts = adata.obs[cell_type_col].value_counts()
    celltype_labels = [f"{ct} ({celltype_counts[ct]})" for ct in celltype_order]

    # Reorder matrices
    median_ordered = median_expr.loc[celltype_order, :].iloc[:, factor_order]
    delta_ordered = delta_age.loc[celltype_order, :].iloc[:, factor_order]

    # Plot setup
    fig, ax = plt.subplots(figsize=(10, 7))
    n_cells = len(median_ordered)
    n_factors = len(median_ordered.columns)
    ax.set_xlim(-0.5, n_factors - 0.5)
    ax.set_ylim(-0.5, n_cells - 0.5)

    # Color scale for delta PSI
    cmap = plt.cm.BrBG
    vmax = max(abs(delta_ordered.values.min()), abs(delta_ordered.values.max()))
    norm = plt.Normalize(vmin=-vmax, vmax=vmax)

    # Draw tiles
    for i, cell_type in enumerate(median_ordered.index):
        for j, factor in enumerate(median_ordered.columns):
            delta = delta_ordered.loc[cell_type, factor]
            color = cmap(norm(delta))
            tile = Rectangle((j - 0.5, i - 0.5), 1, 1, facecolor=color, edgecolor='none')  # pyright: ignore[reportUndefinedVariable]
            ax.add_patch(tile)

    # Draw circle outlines for activity
    min_radius = 0.1
    for i, cell_type in enumerate(median_ordered.index):
        for j, factor in enumerate(median_ordered.columns):
            expr = median_ordered.loc[cell_type, factor]
            if expr > 0.001:
                radius = max(min_radius, (np.log10(expr + 0.01) + 2) * 0.15)
                delta = delta_ordered.loc[cell_type, factor]
                color = cmap(norm(delta))  # match background
                circle = Circle((j, i), radius, facecolor=color, edgecolor='black', linewidth=0.4)
                ax.add_patch(circle)

    # Axis labels
    ax.set_xticks(range(n_factors))
    ax.set_xticklabels(median_ordered.columns, rotation=90, fontsize=10)
    ax.set_yticks(range(n_cells))
    ax.set_yticklabels(celltype_labels, fontsize=10)
    ax.set_aspect('equal')
    ax.tick_params(left=False, bottom=False)
    ax.grid(False)

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, label='Δ SP Activity (old - young)', shrink=0.3)

    from matplotlib.offsetbox import AnchoredOffsetbox, AuxTransformBox, VPacker, HPacker, TextArea

    # Legend values (representative activity levels)
    example_exprs = [0.01, 0.1, 1.0]

    # Function to compute radius in axis units
    def compute_radius(val):
        return max(min_radius, (np.log10(val + 0.01) + 2) * 0.15)

    # Build legend items
    legend_items = []
    for val in example_exprs:
        radius = compute_radius(val)

        # Circle scaled correctly in data units
        circle_box = AuxTransformBox(ax.transData)
        circle = Circle((0, 0), radius, facecolor='white', edgecolor='black', linewidth=0.6)
        circle_box.add_artist(circle)

        # Label
        label = TextArea(f"{val:.2f}", textprops=dict(fontsize=8, va='center', ha='left'))

        # Combine horizontally: [circle | label]
        item = HPacker(children=[circle_box, label], align="center", pad=0, sep=5)
        legend_items.append(item)

    # Combine all items vertically
    legend_box = VPacker(children=legend_items, align="left", pad=0, sep=6)

    # Place the anchored box
    anchored_legend = AnchoredOffsetbox(
        loc='upper right',
        child=legend_box,
        frameon=True,
        bbox_to_anchor=(1.25, 1.0),
        bbox_transform=ax.transAxes,
        borderpad=0.5,
        pad=0.5
    )
    ax.add_artist(anchored_legend)

    plt.tight_layout()
    return fig

In [ ]:
splice_adata

In [ ]:
cell_type_col = "broad_cell_type"
fig = plot_factor_circles(splice_adata, cell_type_col=cell_type_col)
plt.savefig(f"{OUTPUT_DIR}/delta_median_old_minus_young_by_{cell_type_col}_dot_heatmap.pdf", format="pdf", bbox_inches='tight')
print(f"Saved to {OUTPUT_DIR}/delta_median_old_minus_young_by_{cell_type_col}_dot_heatmap.pdf")

In [ ]:
cell_type_col = "tissue_celltype"
fig = plot_factor_circles(splice_adata[splice_adata.obs["tissue_celltype"].isin(median_order)], cell_type_col=cell_type_col)
plt.savefig(f"{OUTPUT_DIR}/delta_median_old_minus_young_by_{cell_type_col}_dot_heatmap.pdf", format="pdf", bbox_inches='tight')
print(f"Saved to {OUTPUT_DIR}/delta_median_old_minus_young_by_{cell_type_col}_dot_heatmap.pdf")

In [ ]:
cell_type_col = "tissue"
fig = plot_factor_circles(splice_adata, cell_type_col=cell_type_col)
plt.savefig(f"{OUTPUT_DIR}/delta_median_old_minus_young_by_{cell_type_col}_dot_heatmap.pdf", format="pdf", bbox_inches='tight')
print(f"Saved to {OUTPUT_DIR}/delta_median_old_minus_young_by_{cell_type_col}_dot_heatmap.pdf")

In [ ]:
# Remove old columns from ge_adata.var
ge_adata.var = ge_adata.var.drop(columns=['Aging_gene', 'is_aging'], errors='ignore')

In [ ]:
# Prepare the aging dataframe with uppercase symbols for matching
aging['symbol_upper'] = aging['symbol'].str.upper()

# Create mapping dictionaries
name_map = aging.set_index('symbol_upper')['name'].to_dict()
lifespan_effect_map = aging.set_index('symbol_upper')['lifespan effect'].to_dict()
longevity_influence_map = aging.set_index('symbol_upper')['longevity influence'].to_dict()
avg_lifespan_change_map = aging.set_index('symbol_upper')['avg lifespan change (max obsv)'].to_dict()
# Create uppercase gene names for mapping
ge_adata.var['gene_name_upper'] = ge_adata.var['gene_name'].str.upper()

# Create the aging_gene boolean column (True if gene is in the aging list)
ge_adata.var['aging_gene'] = ge_adata.var['gene_name_upper'].isin(aging['symbol_upper'])

# Map the other columns
ge_adata.var['aging_gene_name'] = ge_adata.var['gene_name_upper'].map(name_map)
ge_adata.var['aging_lifespan_effect'] = ge_adata.var['gene_name_upper'].map(lifespan_effect_map)
ge_adata.var['aging_longevity_influence'] = ge_adata.var['gene_name_upper'].map(longevity_influence_map)
ge_adata.var['aging_avg_lifespan_change'] = ge_adata.var['gene_name_upper'].map(avg_lifespan_change_map)

# Check results
print(f"Total aging genes mapped: {ge_adata.var['aging_gene'].sum()}")
print(f"Total genes in ge_adata: {len(ge_adata.var)}")
print(f"Percentage of aging genes: {ge_adata.var['aging_gene'].sum() / len(ge_adata.var) * 100:.2f}%")

In [ ]:
print(f"\nSample of mapped aging genes:")
print(ge_adata.var[ge_adata.var['aging_gene']][['gene_name', 'aging_gene_name', 'aging_lifespan_effect', 'aging_longevity_influence', 'aging_avg_lifespan_change']].head(30))

In [ ]:
# reset index in ge_adata.var
ge_adata.var = ge_adata.var.reset_index(drop=True)

In [ ]:
def plot_factor_junction_annotation_counts(final_df, output_path):
    counts = (
        final_df.groupby(["factor_idx", "junction_annotation"])["junction_id_index"]
        .nunique()
        .reset_index(name="count")
    )
    counts["factor_name"] = "SP_" + (counts["factor_idx"] + 1).astype(str)
    
    # Ensure ordering by SP_1, SP_2, SP_3, ...
    # Create the desired order based on sorted unique factor_idx values
    factor_idxs_sorted = sorted(counts["factor_idx"].unique())
    factor_name_order = ["SP_" + str(idx + 1) for idx in factor_idxs_sorted]

    pivot_df = counts.pivot(index="factor_name", columns="junction_annotation", values="count").fillna(0)
    # Reindex to desired order
    pivot_df = pivot_df.reindex(factor_name_order)
    pivot_df["total"] = pivot_df.sum(axis=1)
    #pivot_df = pivot_df.sort_values("total", ascending=False)
    pivot_df = pivot_df.drop(columns="total")

    # --- PLOTTING CHANGES START HERE ---

    # Add figsize directly to the plot call
    ax = pivot_df.plot(
        kind="bar",
        stacked=True,
        colormap="viridis",
        edgecolor="black",
        figsize=(6, 3) 
    )

    # The rest of your customizations stay the same
    plt.xlabel("Splicing Program", fontsize=11)
    plt.ylabel("# of Significant Junctions", fontsize=11)

    # Center the 90-degree rotated tick labels with the bar
    # Set rotation and horizontal alignment, and vertical alignment to center with bar
    plt.xticks(rotation=90, ha="center", va="center", fontsize=11)
    plt.yticks(fontsize=11)

    # Note: pandas plot returns the axes 'ax', you could also use ax.legend(...)
    plt.legend(title="Annotation", fontsize=9, title_fontsize=10, bbox_to_anchor=(1.02, 1), loc="upper left")
    
    # Tight layout is crucial when altering aspect ratio significantly
    plt.tight_layout()

    # --- PLOTTING CHANGES END HERE ---

    plt.savefig(output_path, dpi=300)
    print(f"Saved to {output_path}")
    plt.show()
    plt.close()

    print(counts)

In [ ]:
plot_factor_junction_annotation_counts(diff_spl, os.path.join(OUTPUT_DIR, "differential_splicing_annotation_type_barplot.pdf"))

In [ ]:
def plot_junction_factor_distribution(final_df, output_path):
    """Create an improved histogram showing distribution of factors per junction"""
    # Get counts of factors/junction
    junc_counts = final_df.groupby("junction_id_index")["factor_idx"].nunique()
    junc_counts_df = pd.DataFrame({"junction_id_index": junc_counts.index, "num_factors": junc_counts.values})
    # Create figure with appropriate size
    plt.figure(figsize=(5, 4))
    # Create histogram with better styling
    ax = sns.histplot(junc_counts_df["num_factors"], bins=range(1, junc_counts_df["num_factors"].max() + 2), 
                     kde=False, color='steelblue', edgecolor='darkblue', alpha=0.7)
    # Add mean line
    mean_factors = junc_counts_df["num_factors"].mean()
    plt.axvline(mean_factors, color='red', linestyle='--', linewidth=2)
    plt.text(mean_factors + 0.2, plt.ylim()[1]*0.9, f'Mean: {mean_factors:.2f}', 
             color='red', fontweight='bold')
    # Add title and labels with better formatting
    plt.xlabel("Number of Splicing Programs per Junction", fontsize=12)
    plt.ylabel("# of junctions", fontsize=12)
    # Add grid for better readability
    plt.grid(axis='y', alpha=0.3)
    # increase tick font size 
    # Adjust layout
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    plt.tight_layout()
    # Save figure with higher DPI
    plt.savefig(output_path, dpi=300)
    print(f"Saved to {output_path}")
    plt.show()
    plt.close()

    print(junc_counts)

In [ ]:
plot_junction_factor_distribution(diff_spl, os.path.join(OUTPUT_DIR, "differential_splicing_barplot.pdf"))

## Recalculate differential splicing junctions

In [ ]:
splice_adata

In [ ]:
# Reset junction_id_index column 
# Note: these have been reordered by the factor order above so should be correct to use for everything downstream now. 
# this is already reordered 
psi_samples = leaflet_model["psi_samples"]
splice_adata.var["junction_id_index"] = np.arange(splice_adata.n_vars)
print("\n>> Running analysis functions...")
results = ds.analyze_all_factors_psi(psi_samples, top_junctions=splice_adata.var["junction_id_index"].values, min_effect_size=0.25)
all_results = []

for factor_idx, (effect_size_list, significance_df) in results.items():
    # Append to list
    all_results.append(significance_df)
    print("\n>> Running analysis functions...")

# Concatenate all results into a single DataFrame
final_df = pd.concat(all_results, ignore_index=True)
final_df["junction_id_index"] = final_df["junction_idx"]
# convert splice_adata.var["junction_id_index"] to dtype int64
splice_adata.var["junction_id_index"] = splice_adata.var["junction_id_index"].astype("int64")

In [ ]:
final_df[final_df["factor_idx"] == 3].sort_values(by="effect_size", ascending=True).head(10)

In [ ]:
# Get junction usage across factors of junction 29467 index 
junc_psi = splice_adata.varm["psi_learned"][68808, :]
# convert to dataframe and add "SP_1" to SPN... and make barplot
junc_psi_df = pd.DataFrame(junc_psi)
# Rename main column to PSI 
junc_psi_df.columns = ["PSI"]
junc_psi_df["factor"] = [f"SP_{i+1}" for i in range(len(junc_psi_df))]
sns.barplot(x="factor", y="PSI", data=junc_psi_df)
# rotate x-axis labels
plt.xticks(rotation=90)
plt.show()

In [ ]:
# now subset splice_adata to only include this junction index 29467 and get values from denoised_PSI layer 
splice_adata_junc = splice_adata[:, splice_adata.var["junction_id_index"] == 68808
]
# Explicitly convert denoised_PSI from ArrayView to numpy array
junc_PSI = np.array(splice_adata_junc.layers["denoised_PSI"]).flatten()
# add this to splice_adata_junc.obs 
splice_adata_junc.obs["PSI"] = junc_PSI
# Plot violin plot of PSI values across age_numeric values
sns.violinplot(x="age_numeric", y="PSI", data=splice_adata_junc.obs)
plt.show()

In [ ]:
# For large datasets, a 2D histogram (hexbin or hist2d) is much faster and more stable than kdeplot.
# Here, we use matplotlib's hexbin for speed with 200,000 points.
sns.kdeplot(splice_adata.obs,
    x= "GAG_score",
    y= "SP_4"
)


In [ ]:
# For large datasets, a 2D histogram (hexbin or hist2d) is much faster and more stable than kdeplot.
# Here, we use matplotlib's hexbin for speed with 200,000 points.
sns.kdeplot(splice_adata.obs,
    x= "GAG_score",
    y= "SP_2"
)

In [ ]:
# Merge with adata.var using junction_id_index
final_df = final_df.merge(splice_adata.var, on="junction_id_index")

In [ ]:
# find common columns between final_df and atse_df to merge on 
common_columns = [col for col in final_df.columns if col in atse_df.columns]
final_df = final_df.merge(atse_df, on=["junction_id", "event_id"])

In [ ]:
final_df[final_df["junction_id"] == "chr10_76892656_76961747_-"]

In [ ]:
final_df.sort_values(by="abs_effect_size", ascending=False).iloc[0]["event_id"]

In [ ]:
mouse_event = "ENSMUSG00000031851.14_atse_1"
# need to fix by adding back columns to atse_df 
visualize_atse_event(mouse_event, atse_df, db_mouse,
                 species="mouse", base_width=7, 
                       trans_height=0.2, 
                  padding=200, show_junc_lines=False, 
                  filter_ensembl_transcripts=True)

In [ ]:
# save final_df to csv 
final_df.to_csv(f"{OUTPUT_DIR}/final_df.tsv.gz", index=False, sep="\t", compression="gzip")
print(f"Saved final_df to {OUTPUT_DIR}/final_df.tsv.gz")

## Check FGFR2 splicing 

In [ ]:
splice_adata.var[splice_adata.var["gene_name"] == "FGFR2"]

In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Update these lists based on your new "broad_cell_type" labels
epi_types = [
    "Epithelial", "Stem/Progenitor" # Including progenitors with epithelial potential
]

mes_types = [
    "Stromal", "Endothelial", "Muscle", "Muscle_Satellite"
]

# We'll also define neural/immune to ensure they get colors too
other_types = [
    "Neuron_Excitatory", "Neuron_Inhibitory", "Immune", "Glia", "Microglia", "Neuron_TMS"
]

# 2. Define colors based on broad categories
def get_color(ct):
    if ct in epi_types: return "goldenrod"
    if ct in mes_types: return "steelblue"
    return "lightgrey" # Fallback for Immune/Neural/Other

# --- pick ATSE event ---
event_id = "ENSMUSG00000030849.18_atse_4"

# --- get junctions for this ATSE ---
junc_meta = splice_adata.var.loc[splice_adata.var["event_id"] == event_id]
j_indices = junc_meta["junction_id_index"].astype(int).tolist()

# --- prep cell type info ---
# Check if your column is named 'broad_cell_type' or 'medium_cell_type'
# Based on your text, 'broad_cell_type' seems to be the category column.
obs_col = "broad_cell_type" 

cell_types = splice_adata.obs[obs_col].astype("category")
keep_types = epi_types + mes_types + other_types
mask = cell_types.isin(keep_types)
cell_types_filtered = cell_types[mask].cat.remove_unused_categories()

# Generate palette for only the categories present
unique_types = cell_types_filtered.unique()
palette = {ct: get_color(ct) for ct in unique_types}

# --- loop through junctions ---
n = len(j_indices)
if n == 0:
    print(f"No junctions found for event {event_id}")
else:
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows), squeeze=False)

    for idx, (ax, j) in enumerate(zip(axes.ravel(), j_indices)):
        # Extract PSI data
        col = splice_adata.layers["denoised_PSI"][:, j]
        junction_id = splice_adata.var["junction_id"].iloc[j]
        
        # Handle sparse vs dense arrays
        psi = col.A1 if hasattr(col, "A1") else col.flatten()
        
        # Build plotting dataframe
        df = pd.DataFrame({
            "PSI": psi[mask], 
            "cell_type": cell_types_filtered
        })

        # Order cell types by decreasing median PSI
        order = (
            df.groupby("cell_type", observed=True)["PSI"]
              .median()
              .sort_values(ascending=False)
              .index.tolist()
        )

        sns.violinplot(
            data=df, x="cell_type", y="PSI",
            inner="box", cut=0, palette=palette, order=order, ax=ax
        )
        
        ax.set_title(f"Junction {junction_id}")
        ax.set_xlabel("")
        ax.set_ylabel("PSI")
        ax.tick_params(axis='x', rotation=90)

    # hide empty subplots
    for ax in axes.ravel()[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# --- pick ATSE event ---
event_id = "ENSMUSG00000030849.18_atse_4"

In [ ]:
import math
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- 1. Define Categories and Colors ---
epi_types = ["Epithelial", "Stem/Progenitor"]
mes_types = ["Stromal", "Endothelial", "Muscle", "Muscle_Satellite"]
other_types = ["Neuron_Excitatory", "Neuron_Inhibitory", "Immune", "Glia", "Microglia", "Neuron_TMS"]

def get_color(ct):
    if ct in epi_types: return "goldenrod"
    if ct in mes_types: return "steelblue"
    return "lightgrey"

# --- 2. Pick ATSE Event ---
event_id = "ENSMUSG00000030849.18_atse_4"

# Validate event existence
if event_id not in splice_adata.var["event_id"].values:
    print(f"Error: {event_id} not found in splice_adata.var['event_id']")
else:
    # --- 3. Get Junction Indices ---
    junc_meta = splice_adata.var.loc[splice_adata.var["event_id"] == event_id]
    j_indices = junc_meta["junction_id_index"].astype(int).tolist()

    # --- 4. Prep Cell Type Info ---
    obs_col = "broad_cell_type" 
    keep_types = epi_types + mes_types + other_types
    
    # Create mask and filtered series
    mask = splice_adata.obs[obs_col].isin(keep_types)
    cell_types_filtered = splice_adata.obs.loc[mask, obs_col].astype("category").cat.remove_unused_categories()

    # Dynamic palette
    palette = {ct: get_color(ct) for ct in cell_types_filtered.unique()}

    # --- 5. Plotting ---
    n = len(j_indices)
    ncols = 3
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows), squeeze=False)

    for idx, (ax, j) in enumerate(zip(axes.ravel(), j_indices)):
        # Extract PSI data safely (handling sparse or dense)
        # .toarray() ensures compatibility if the layer is a sparse matrix
        col_data = splice_adata.layers["denoised_PSI"][:, j]
        
        if hasattr(col_data, "toarray"):
            psi = col_data.toarray().ravel()
        elif hasattr(col_data, "A1"):
            psi = col_data.A1
        else:
            psi = np.asarray(col_data).ravel()
        
        # Build plotting dataframe - specifically using the boolean mask values
        df = pd.DataFrame({
            "PSI": psi[mask.values], 
            "cell_type": cell_types_filtered.values
        })

        # Drop NaNs to prevent violinplot errors
        df = df.dropna(subset=["PSI"])

        # Order cell types by decreasing median PSI
        if not df.empty:
            order = (
                df.groupby("cell_type", observed=True)["PSI"]
                  .median()
                  .sort_values(ascending=False)
                  .index.tolist()
            )

            sns.violinplot(
                data=df, x="cell_type", y="PSI",
                inner="box", cut=0, palette=palette, order=order, ax=ax
            )
        
        junction_id = splice_adata.var["junction_id"].iloc[j]
        ax.set_title(f"Junction {junction_id}")
        ax.set_xlabel("")
        ax.set_ylabel("PSI")
        ax.tick_params(axis='x', rotation=90)

    # Clean up empty subplots
    for ax in axes.ravel()[n:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
output_dir="/gpfs/commons/home/kisaev/Leaflet-analysis/LeafletFA_Submission2025/Mouse_Splicing_Foundation/Figures/FIGURE2/plots"

In [ ]:
import math

# --- 1. Settings & Specific Mapping ---
event_id = "ENSMUSG00000030849.18_atse_4"
# Using the specific tissue_celltype column as requested
obs_col = "tissue_celltype" 

# Extracted from your provided list for high-precision grouping
epi_specific = [
    "Skin_Keratinocyte", "Tongue_Keratinocyte", "LI_IntEpi", "Mammary_Epithelial",
    "Bladder_Urothelial", "LI_EpiProg", "Pancreas_Islet", "Liver_Hepatocyte",
    "Trachea_Epithelial", "Kidney_RenalEpi", "Lung_Pneumocyte", "LI_Secretory",
    "Mammary_GlandEpi", "Lung_Epithelial", "Trachea_Secretory", "BAT_Epithelial",
    "GAT_Epithelial", "MAT_Epithelial", "SCAT_Epithelial", "Pancreas_Acinar",
    "Lung_Neuroendo", "Kidney_RenalEpi", "Brain_Epithelial_(CNS"
]

mes_specific = [
    "Heart_Fibroblast", "Heart_VascEndo", "Lung_SmoothMus", "Muscle_Satellite",
    "Brain_Endothelial", "GAT_MSC", "SCAT_MSC", "MAT_MSC", "Trachea_Fibroblast",
    "Muscle_MSC", "Diaphragm_Satellite", "Liver_SinEndo", "Lung_Fibroblast",
    "Muscle_Endothelial", "SCAT_Endothelial", "Aorta_VascEndo", "Heart_Endothelial",
    "Brain_Pericyte", "GAT_Endothelial", "BAT_Endothelial", "Lung_Vascular",
    "Trachea_Cartilage", "Kidney_VascEndo", "BAT_MSC", "Lung_VascEndo",
    "Heart_SmoothMus", "Lung_Lymphatic", "Diaphragm_MSC", "Heart_Muscle",
    "Kidney_Stromal", "Aorta_Fibroblast", "Pancreas_Endothelial", "Trachea_Endothelial",
    "Diaphragm_Endothelial", "Mammary_Endothelial", "Brain_VascStroma",
    "Pancreas_Fibroblast", "Aorta_Fibroblast", "Kidney_Fibroblast", "Skin_Fibroblast"
]

# --- 2. Data Extraction ---
if event_id not in splice_adata.var["event_id"].values:
    print(f"Error: {event_id} not found.")
else:
    junc_meta = splice_adata.var.loc[splice_adata.var["event_id"] == event_id]
    j_indices = junc_meta["junction_id_index"].astype(int).tolist()
    j_names = junc_meta["junction_id"].tolist()

    all_data = []
    for j_idx, j_id in zip(j_indices, j_names):
        col_data = splice_adata.layers["denoised_PSI"][:, j_idx]
        psi = col_data.toarray().ravel() if hasattr(col_data, "toarray") else np.asarray(col_data).ravel()
        
        df_j = pd.DataFrame({
            "PSI": psi,
            "specific_type": splice_adata.obs[obs_col].values,
            "Junction": j_id
        })
        all_data.append(df_j)

    df_combined = pd.concat(all_data)

    # --- 3. Lineage Mapping Logic ---
    def map_lineage(x):
        x_clean = str(x).strip()
        if x_clean in epi_specific: return "Epithelial"
        if x_clean in mes_specific: return "Mesenchymal"
        return None

    df_combined["MajorGroup"] = df_combined["specific_type"].apply(map_lineage)
    df_plot = df_combined.dropna(subset=["MajorGroup", "PSI"]).copy()

    # --- 4. Count Unique Cells ---
    n_juncs = len(j_names)
    counts = df_plot.groupby("MajorGroup", observed=True).size() // n_juncs
    
    group_order = ["Epithelial", "Mesenchymal"]
    new_labels = [f"{grp}\n(N={counts.get(grp, 0)})" for grp in group_order]

    # --- 5. Final Formatting & Plotting ---
    fig, ax = plt.subplots(figsize=(4,5))

    # Using colors from image_0e47e7.png
    sns.violinplot(
        data=df_plot, 
        x="MajorGroup", 
        y="PSI", 
        hue="Junction", 
        order=group_order,
        split=False, 
        inner="box", 
        cut=0, 
        palette=["steelblue", "goldenrod"],
        ax=ax
    )

    ax.set_xticks(range(len(group_order)))
    ax.set_xticklabels(new_labels, rotation=0, fontsize=14)

    ax.set_xlabel("Cell Lineage", fontsize=16, labelpad=10)
    ax.set_ylabel(r"Imputed $\Psi$", fontsize=16)
    ax.tick_params(axis='both', labelsize=14)
    ax.tick_params(axis='x', length=0)

    # Legend at the bottom
    ax.legend(title="Junction", loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=1)

    plt.tight_layout()
    # Save figure 
    plt.savefig(f"{output_dir}/specific_tissue_celltype_violinplot.pdf", format="pdf", bbox_inches='tight')
    print(f"Saved to {output_dir}/specific_tissue_celltype_violinplot.pdf")
    plt.show()

In [ ]:
# chr7_130196374_130198418_- --> Exon Number 8
# chr7_130196374_130199757_- --> Exon Number 6? earlier than 8 

In [ ]:
output_dir

In [ ]:
mouse_event = "ENSMUSG00000030849.18_atse_4"
# need to fix by adding back columns to atse_df 
visualize_atse_event(mouse_event, atse_df, db_mouse,
                 species="mouse", base_width=8, 
                       trans_height=0.4, output_dir=output_dir,
                  padding=500, show_junc_lines=False)

In [ ]:
splice_adata

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. Define Splicing Program columns
sp_cols = [f'SP_{i}' for i in range(1, 21)]

# 2. Calculate Mean Activity per Broad Cell Type
# This is often more interpretable than regression coefficients for characterization
celltype_means = splice_adata.obs.groupby('broad_cell_type')[sp_cols].mean()

# Calculate Z-scores across cell types to see where programs are "exceptionally" active
celltype_z = (celltype_means - celltype_means.mean()) / celltype_means.std()

# 3. Quick Logistic Regression (One-vs-Rest)
# To see which SPs distinguish each cell type from others
X = splice_adata.obs[sp_cols]
y = splice_adata.obs['broad_cell_type']

# Scale features for regression
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Fit Logistic Regression (One-vs-Rest)
# Using a small C (regularization) to keep coefficients robust
lr = LogisticRegression(multi_class='ovr', solver='lbfgs', max_iter=1000, C=0.1)
lr.fit(X_scaled, y)

# Extract coefficients into a DataFrame
# Rows = Cell Types, Columns = SPs
coef_df = pd.DataFrame(lr.coef_, index=lr.classes_, columns=sp_cols)

# 4. Summarize results for SP15 and SP18
for sp in ['SP_15', 'SP_18']:
    print(f"\n--- Analysis for {sp} ---")
    
    # Top cell types by Mean Activity
    top_means = celltype_means[sp].sort_values(ascending=False).head(5)
    print(f"Top 5 cell types (Mean Activity):\n{top_means}")
    
    # Top cell types by Logistic Regression Association
    top_assoc = coef_df[sp].sort_values(ascending=False).head(5)
    print(f"\nTop 5 cell types (Regression Association):\n{top_assoc}")

# 5. Export results for the paper
celltype_means.to_csv('sp_activity_by_broad_cell_type.csv')
coef_df.to_csv('sp_to_celltype_logistic_regression_coeffs.csv')

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# 1. Define Splicing Program columns
sp_cols = [f'SP_{i}' for i in range(1, 21)]

# 2. Filter for robust groups (n >= 50)
counts = splice_adata.obs['tissue_celltype'].value_counts()
valid_groups = counts[counts >= 50].index
subset_obs = splice_adata.obs[splice_adata.obs['tissue_celltype'].isin(valid_groups)]

# 3. Calculate Mean Activity per tissue_celltype
tc_means = subset_obs.groupby('tissue_celltype')[sp_cols].mean()

# 4. Logistic Regression (One-vs-Rest) at tissue_celltype level
X_tc = subset_obs[sp_cols]
y_tc = subset_obs['tissue_celltype']

scaler = StandardScaler()
X_tc_scaled = scaler.fit_transform(X_tc)

# Using L1 penalty (liblinear) can help identify sparse, specific associations
lr_tc = LogisticRegression(multi_class='ovr', solver='liblinear', penalty='l1', C=0.5)
lr_tc.fit(X_tc_scaled, y_tc)

coef_tc_df = pd.DataFrame(lr_tc.coef_, index=lr_tc.classes_, columns=sp_cols)

# 5. Summarize SP15 and SP18
for sp in ['SP_15', 'SP_18']:
    print(f"\n--- Tissue-CellType Analysis for {sp} ---")
    
    # Top 10 for more detail
    top_tc_assoc = coef_tc_df[sp].sort_values(ascending=False).head(10)
    print(f"Top 10 Tissue-CellType Associations (Regression):\n{top_tc_assoc}")

# Save for detailed lookup
coef_tc_df.to_csv('sp_to_tissue_celltype_coeffs.csv')